# Merge Tables


## Intro


_Developer Note:_ if you may make a PR in the future, be sure to copy this
notebook, and use the `gitignore` prefix `temp` to avoid future conflicts.

This is one notebook in a multi-part series on Spyglass.

- To set up your Spyglass environment and database, see
  [the Setup notebook](./00_Setup.ipynb)
- To insert data, see [the Insert Data notebook](./01_Insert_Data.ipynb)
- For additional info on DataJoint syntax, including table definitions and
  inserts, see
  [these additional tutorials](https://github.com/datajoint/datajoint-tutorials)
- For information on why we use merge tables, and how to make one, see our
  [documentation](https://lorenfranklab.github.io/spyglass/0.4/misc/merge_tables/)

In short, merge tables represent the end processing point of a given way of
processing the data in our pipelines. Merge Tables allow us to build new
processing pipeline, or a new version of an existing pipeline, without having to
drop or migrate the old tables. They allow data to be processed in different
ways, but with a unified end result that downstream pipelines can all access.


## Imports


Let's start by importing the `spyglass` package, along with a few others.


In [1]:
import datajoint as dj

dj.config.load("../dj_local_conf_1335.json")
dj.conn()

import warnings

warnings.simplefilter("ignore", category=DeprecationWarning)
warnings.simplefilter("ignore", category=ResourceWarning)
warnings.simplefilter("ignore", category=UserWarning)

import spyglass.common as sgc
import spyglass.lfp as lfp
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename
from spyglass.utils.dj_merge_tables import Merge
from spyglass.lfp.lfp_merge import LFPOutput  # Merge Table

[2026-04-27 19:07:57,482][INFO]: DataJoint is configured from /home/cb/wrk/spyglass/1335/dj_local_conf_1335.json
[2026-04-27 19:07:57,486][INFO]: DataJoint 0.14.6 connected to root@localhost:10899


## Example data


Check to make sure the data inserted in the previour notebook is still there.


In [2]:
nwb_file_name = "minirec20230622.nwb"
nwb_copy_file_name = get_nwb_copy_filename(nwb_file_name)
nwb_file_dict = {"nwb_file_name": nwb_copy_file_name}
sgc.Session & nwb_file_dict

nwb_file_name name of the NWB file,subject_id,institution_name,lab_name,session_id,session_description,session_start_time,timestamps_reference_time,experiment_description
minirec20230622_.nwb,54321,UCSF,Loren Frank Lab,12345,test yaml insertion,2023-06-22 15:59:58,1970-01-01 00:00:00,Test Conversion


If you haven't already done so, insert data into a Merge Table.

_Note_: Some existing parents of Merge Tables perform the Merge Table insert as
part of the populate methods. This practice will be revised in the future.

<!-- TODO: Add entry to another parent to cover mutual exclusivity issues. -->


In [3]:
sgc.FirFilterParameters().create_standard_filters()
lfp.lfp_electrode.LFPElectrodeGroup.create_lfp_electrode_group(
    nwb_file_name=nwb_copy_file_name,
    group_name="test",
    electrode_list=[0],
    skip_duplicates=True,
)
lfp_key = {
    "nwb_file_name": nwb_copy_file_name,
    "lfp_electrode_group_name": "test",
    "target_interval_list_name": "01_s1",
    "filter_name": "LFP 0-400 Hz",
    "filter_sampling_rate": 30_000,
}
lfp.v1.LFPSelection.insert1(lfp_key, skip_duplicates=True)
lfp.v1.LFPV1().populate(lfp_key)
LFPOutput.insert([lfp_key], skip_duplicates=True)

[19:08:01][INFO] Spyglass: Key already in part LFPV1: {'nwb_file_name': 'minirec20230622_.nwb', 'lfp_electrode_group_name': 'test', 'target_interval_list_name': '01_s1', 'filter_name': 'LFP 0-400 Hz', 'filter_sampling_rate': 30000}


## Helper functions


Merge Tables expose **instance methods** that operate on a restricted table.
The older `merge_*` class methods are deprecated (Spyglass 0.7.0) but still
work and emit a warning on first call.

<details>
<summary><b>Familiar with the old API? Click to see the migration table.</b></summary>

| Old call | New equivalent |
|---|---|
| `T.merge_view(r)` | `(T & r).view()` |
| `T.merge_html(r)` | `(T & r).html()` |
| `T.merge_restrict(r)` | `T & r` |
| `T.merge_fetch(r, *attrs)` | `(T & r).fetch(*attrs)` |
| `T.merge_get_part(r, ...)` | `(T & r).get_part_table(...)` |
| `T.merge_get_parent(r, ...)` | `(T & r).get_parent_table(...)` |
| `T.merge_delete(r)` | `(T & r).delete()` |
| `T.merge_delete_parent(r, dry_run=True)` | `(T & r).delete_upstream(dry_run=True)` |
| `T.merge_populate(source, keys)` | `T.populate(source, keys)` |

All `merge_*` methods still work but will be removed in Spyglass 0.7.0.

</details>

In [4]:
# New instance methods (preferred)
new_methods = [
    "view",
    "html",
    "get_part_table",
    "get_parent_table",
    "fetch",
    "fetch1",
    "super_fetch",
    "delete",
    "delete_upstream",
    "populate",
]
print("New API:", new_methods)

# Deprecated class methods (still work, emit warnings)
deprecated = [d for d in dir(Merge) if d.startswith("merge_")]
print("Deprecated:", deprecated)

New API: ['view', 'html', 'get_part_table', 'get_parent_table', 'fetch', 'fetch1', 'super_fetch', 'delete', 'delete_upstream', 'populate']
Deprecated: ['merge_delete', 'merge_delete_parent', 'merge_fetch', 'merge_get_parent', 'merge_get_parent_class', 'merge_get_part', 'merge_html', 'merge_populate', 'merge_restrict', 'merge_restrict_class', 'merge_view']


In [5]:
help(LFPOutput.get_part_table)

Help on function get_part_table in module spyglass.utils.dj_merge_tables:

get_part_table(self, join_master: bool = False, restrict_part: bool = True, multi_source: bool = False, return_empties: bool = False)
    Return part table(s) for self.restriction.

    Instance-method replacement for ``merge_get_part``. Use as
    ``(T & restriction).get_part_table()``.



## Showing data


Merge tables behave like they're already joined with the parts.


In [6]:
%load_ext autoreload
%autoreload 2

In [7]:
LFPOutput()

merge_id,source,nwb_file_name name of the NWB file,lfp_electrode_group_name name for this group of electrodes,target_interval_list_name descriptive name of this interval list,filter_name descriptive name of this filter,filter_sampling_rate sampling rate for this filter
1b37c6a9-61eb-9f7c-c974-664a62993f8e,LFPV1,minirec20230622_.nwb,test,01_s1,LFP 0-400 Hz,30000
a4c79bfe-f398-c528-880e-fa6e058a9773,LFPV1,minirec20230622_.nwb,test 2,01_s1,LFP 0-400 Hz,30000


This also works with `dj.Top`

In [8]:
LFPOutput & dj.Top()

merge_id,source,nwb_file_name name of the NWB file,lfp_electrode_group_name name for this group of electrodes,target_interval_list_name descriptive name of this interval list,filter_name descriptive name of this filter,filter_sampling_rate sampling rate for this filter
1b37c6a9-61eb-9f7c-c974-664a62993f8e,LFPV1,minirec20230622_.nwb,test,01_s1,LFP 0-400 Hz,30000


To get the standard view of the table, use `T.super_view()`

In [9]:
LFPOutput().super_view()

merge_id,source
1b37c6a9-61eb-9f7c-c974-664a62993f8e,LFPV1
a4c79bfe-f398-c528-880e-fa6e058a9773,LFPV1


Restrict Merge Tables with the standard `&` operator. Both dict and string
restrictions that reference part-table fields are **resolved automatically**:

```python
Merge & {"field": "value"}      # ✓ dict — resolves through parts
Merge & 'field LIKE "val%"'     # ✓ string — also resolves through parts
Merge & {"merge_id": some_uuid} # ✓ master field — unchanged
```

In [10]:
LFPOutput & nwb_file_dict

merge_id,source,nwb_file_name name of the NWB file,lfp_electrode_group_name name for this group of electrodes,target_interval_list_name descriptive name of this interval list,filter_name descriptive name of this filter,filter_sampling_rate sampling rate for this filter
1b37c6a9-61eb-9f7c-c974-664a62993f8e,LFPV1,minirec20230622_.nwb,test,01_s1,LFP 0-400 Hz,30000
a4c79bfe-f398-c528-880e-fa6e058a9773,LFPV1,minirec20230622_.nwb,test 2,01_s1,LFP 0-400 Hz,30000


`fetch()` walks parts → returns part-table data

In [11]:
(LFPOutput & nwb_file_dict).fetch(as_dict=True)[0]

{'merge_id': UUID('1b37c6a9-61eb-9f7c-c974-664a62993f8e'),
 'nwb_file_name': 'minirec20230622_.nwb',
 'lfp_electrode_group_name': 'test',
 'target_interval_list_name': '01_s1',
 'filter_name': 'LFP 0-400 Hz',
 'filter_sampling_rate': 30000}

`fetch1()` follows DataJoint convention

In [12]:
restrict = LFPOutput & "lfp_electrode_group_name='test'"
restrict.fetch1()

{'merge_id': UUID('1b37c6a9-61eb-9f7c-c974-664a62993f8e'),
 'nwb_file_name': 'minirec20230622_.nwb',
 'lfp_electrode_group_name': 'test',
 'target_interval_list_name': '01_s1',
 'filter_name': 'LFP 0-400 Hz',
 'filter_sampling_rate': 30000}

In [13]:
LFPOutput.fetch("filter_name")

array(['LFP 0-400 Hz', 'LFP 0-400 Hz'], dtype=object)

In [14]:
LFPOutput.fetch("filter_name", "lfp_sampling_rate")

[19:08:02][WARNING] Spyglass: Attribute `lfp_sampling_rate` not found. Skipping LFPV1
[19:08:02][INFO] Spyglass: No merge_fetch results.
	If not restricting, try: `M.fetch(True,'attr')
	If restricting by source, use dict: `M.fetch({'source':'X'}


[]

`super_fetch()` stays on master → returns only merge_id and source

In [15]:
(LFPOutput & nwb_file_dict).super_fetch(as_dict=True)[0]

{'merge_id': UUID('1b37c6a9-61eb-9f7c-c974-664a62993f8e'), 'source': 'LFPV1'}

Fetch just the master primary key (bypasses part-walking)

In [16]:
uuid_key = (LFPOutput & nwb_file_dict).fetch("KEY", limit=1)[0]
restrict = LFPOutput & uuid_key
restrict

merge_id,source,nwb_file_name name of the NWB file,lfp_electrode_group_name name for this group of electrodes,target_interval_list_name descriptive name of this interval list,filter_name descriptive name of this filter,filter_sampling_rate sampling rate for this filter
1b37c6a9-61eb-9f7c-c974-664a62993f8e,LFPV1,minirec20230622_.nwb,test,01_s1,LFP 0-400 Hz,30000


Fetch all part-table columns as a dict

In [17]:
nwb_key = (LFPOutput & nwb_file_dict).fetch(as_dict=True)[0]
nwb_key

{'merge_id': UUID('1b37c6a9-61eb-9f7c-c974-664a62993f8e'),
 'nwb_file_name': 'minirec20230622_.nwb',
 'lfp_electrode_group_name': 'test',
 'target_interval_list_name': '01_s1',
 'filter_name': 'LFP 0-400 Hz',
 'filter_sampling_rate': 30000}

In [18]:
LFPOutput().fetch_nwb(nwb_key)[0]

{'nwb_file_name': 'minirec20230622_.nwb',
 'lfp_electrode_group_name': 'test',
 'target_interval_list_name': '01_s1',
 'filter_name': 'LFP 0-400 Hz',
 'filter_sampling_rate': np.int64(30000),
 'analysis_file_name': 'minirec20230622_M8QKFQZAYW.nwb',
 'interval_list_name': 'lfp_test_01_s1_valid times',
 'lfp_object_id': '15a2cc1a-29c6-4db5-93bb-0381858541e7',
 'lfp_sampling_rate': np.float64(1000.0),
 'lfp': filtered data pynwb.ecephys.ElectricalSeries at 0x137692521162432
 Fields:
   comments: no comments
   conversion: 1.0
   data: <HDF5 dataset "data": shape (10476, 1), type "<i2">
   description: filtered data
   electrodes: electrodes <class 'hdmf.common.table.DynamicTableRegion'>
   interval: 1
   offset: 0.0
   resolution: -1.0
   timestamps: <HDF5 dataset "timestamps": shape (10476,), type "<f8">
   timestamps_unit: seconds
   unit: volts}

## Selecting data


- `get_part_table()` returns the Merge Part table for the current restriction.
- `get_parent_table()` returns the upstream source table (where the actual data
lives). 

Both accept `join_master=True` to include `merge_id` / `source`.

In [19]:
result4 = (LFPOutput & nwb_file_dict).get_part_table(join_master=True)
result4

merge_id,source,nwb_file_name name of the NWB file,lfp_electrode_group_name name for this group of electrodes,target_interval_list_name descriptive name of this interval list,filter_name descriptive name of this filter,filter_sampling_rate sampling rate for this filter
1b37c6a9-61eb-9f7c-c974-664a62993f8e,LFPV1,minirec20230622_.nwb,test,01_s1,LFP 0-400 Hz,30000
a4c79bfe-f398-c528-880e-fa6e058a9773,LFPV1,minirec20230622_.nwb,test 2,01_s1,LFP 0-400 Hz,30000


In [23]:
result5 = (LFPOutput & 'nwb_file_name LIKE "mini%"').get_parent_table()
result5

nwb_file_name name of the NWB file,lfp_electrode_group_name name for this group of electrodes,target_interval_list_name descriptive name of this interval list,filter_name descriptive name of this filter,filter_sampling_rate sampling rate for this filter,analysis_file_name name of the file,interval_list_name descriptive name of this interval list,lfp_object_id object ID for loading from the NWB file,"lfp_sampling_rate the sampling rate, in HZ"
minirec20230622_.nwb,test,01_s1,LFP 0-400 Hz,30000,minirec20230622_M8QKFQZAYW.nwb,lfp_test_01_s1_valid times,15a2cc1a-29c6-4db5-93bb-0381858541e7,1000.0
minirec20230622_.nwb,test 2,01_s1,LFP 0-400 Hz,30000,minirec20230622_YLPIYYSQ3D.nwb,lfp_test 2_01_s1_valid times,93132add-aa13-4175-a758-032c86c85eaa,1000.0


In [26]:
result5.full_table_name

'`lfp_v1`.`__l_f_p_v1`'

## Deletion from Merge Tables


When deleting from Merge Tables, we have three options:

1. **`(T & restriction).delete()`** — removes the master and part entries from
   the Merge Table. The upstream source data (e.g., `LFPV1`) is *not* removed.

2. **`(T & restriction).delete_upstream(dry_run=True)`** — removes entries from
   the upstream source tables. Use `dry_run=True` (default) to preview first.

3. **`table.delete_downstream_parts(dry_run=True)`** — available on any
   upstream table; finds downstream Merge Table entries and removes them,
   preventing orphaned master rows.

The latter two are destructive. Call `delete_upstream` before `delete()` if
you want to also remove the upstream source data in the same session.

In [33]:
# Preview: what would delete_upstream remove from the source tables?
(LFPOutput & nwb_file_dict).delete_upstream(dry_run=True)

[FreeTable(`lfp_v1`.`__l_f_p_v1`)
 *nwb_file_name *lfp_electrode *target_interv *filter_name   *filter_sampli analysis_file_ interval_list_ lfp_object_id  lfp_sampling_r
 +------------+ +------------+ +------------+ +------------+ +------------+ +------------+ +------------+ +------------+ +------------+
 minirec2023062 test           01_s1          LFP 0-400 Hz   30000          minirec2023062 lfp_test_01_s1 15a2cc1a-29c6- 1000.0        
  (Total: 1)]

In [34]:
# Delete from the Merge Table (master + parts)
(LFPOutput & nwb_file_dict).delete()

[2026-04-27 16:42:32,510][WARNING]: Delete cancelled


This function is run automatically whin you use `cautious_delete`, which
checks team permissions before deleting.


In [40]:
from spyglass.common import Nwbfile

(Nwbfile & nwb_file_dict).cautious_delete()

[2026-04-27 16:44:42,272][WARNING]: Delete cancelled


## Up Next


In the [next notebook](./10_Spike_Sorting.ipynb), we'll start working with
ephys data with spike sorting.
